# Tahap 2 - Desain Algoritma Genetika

**Problem 12: Manufaktur - Optimasi Penjadwalan Pemeliharaan Mesin**

Pada tahap ini kami merancang cara menggunakan hasil prediksi dari Tahap 1 untuk menyusun urutan perbaikan 50 mesin dengan Algoritma Genetika.

Kami memilih Algoritma Genetika karena solusi yang dicari berupa urutan antrean, bukan satu angka atau satu label. Untuk 50 mesin, jumlah kemungkinan urutan sangat besar jika dicoba satu per satu. Algoritma Genetika membantu mencari jadwal yang baik lewat seleksi, crossover, dan mutasi tanpa harus memeriksa semua kemungkinan urutan.

## 1. Deklarasi Arsitektur Integrasi

Kami memilih **Opsi A (Pra-Komputasi)**. Model Machine Learning memprediksi sisa umur semua mesin satu kali di awal. Setelah itu, Algoritma Genetika menggunakan `rul_ga_jam` dan `durasi_perbaikan_jam` untuk menilai setiap urutan jadwal.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r'D:/tubes_ka')
STAGE1 = ROOT / 'tahap 1'
STAGE2 = ROOT / 'tahap 2'
DATA_PATH = STAGE1 / 'outputs' / 'data' / 'input_50_mesin_anomali_prediksi.csv'
df = pd.read_csv(DATA_PATH)
df['rul_ga_jam'] = df['prediksi_sisa_umur_jam'].clip(lower=0).round(2)
df['machine_code'] = [f'M{i:02d}' for i in range(1, len(df) + 1)]
df[['machine_code', 'id_log_sensor', 'jenis_mesin', 'prediksi_sisa_umur_jam', 'rul_ga_jam', 'durasi_perbaikan_jam']].head(10)

## 2. Desain Kromosom

Kromosom berbentuk **permutasi**. Setiap gen adalah kode mesin. Urutan gen menunjukkan urutan antrean perbaikan yang dikerjakan oleh satu tim teknisi.

In [ ]:
urutan_awal = df.sort_values(['rul_ga_jam', 'durasi_perbaikan_jam']).index.tolist()
kromosom_awal = df.loc[urutan_awal, 'machine_code'].head(10).tolist()
print('10 gen pertama kromosom:')
print(kromosom_awal)

## 3. Fungsi Fitness

Mesin dihitung breakdown jika waktu mulai perbaikannya lebih besar dari sisa umur prediksi. Nilai fitness dibuat dari cost. Semakin sedikit breakdown dan keterlambatan, semakin baik jadwal tersebut.

In [ ]:
def evaluate_schedule(order, data):
    seen = set(order)
    if len(order) != len(data) or len(seen) != len(data) or seen != set(range(len(data))):
        raise ValueError('Kromosom tidak valid: harus berupa permutasi lengkap tanpa duplikasi.')

    time_now = 0.0
    breakdown_count = 0
    total_lateness = 0.0
    total_waiting = 0.0
    schedule_rows = []

    for seq, idx in enumerate(order, start=1):
        row = data.iloc[idx]
        start_time = time_now
        finish_time = start_time + float(row['durasi_perbaikan_jam'])
        rul = float(row['rul_ga_jam'])
        lateness = max(0.0, start_time - rul)
        breakdown = int(start_time > rul)

        breakdown_count += breakdown
        total_lateness += lateness
        total_waiting += start_time
        schedule_rows.append({
            'urutan': seq,
            'machine_code': row['machine_code'],
            'id_log_sensor': row['id_log_sensor'],
            'rul_ga_jam': rul,
            'durasi_perbaikan_jam': row['durasi_perbaikan_jam'],
            'mulai_jam': start_time,
            'selesai_jam': finish_time,
            'breakdown_saat_menunggu': breakdown,
            'keterlambatan_jam': lateness,
        })
        time_now = finish_time

    cost = 10000 * breakdown_count + 100 * total_lateness + 0.01 * total_waiting
    fitness = 1 / (1 + cost)
    return {
        'breakdown_count': breakdown_count,
        'total_lateness_jam': total_lateness,
        'total_waiting_jam': total_waiting,
        'makespan_jam': time_now,
        'cost': cost,
        'fitness': fitness,
        'schedule': pd.DataFrame(schedule_rows),
    }

## 4. Verifikasi Fitness

Tabel berikut menunjukkan hasil penilaian beberapa jadwal pembanding sebelum proses integrasi pada Tahap 3.

In [ ]:
orders = {
    'FCFS_input_awal': list(range(len(df))),
    'SPT_durasi_terpendek': df.sort_values('durasi_perbaikan_jam').index.tolist(),
    'EDD_RUL_terpendek': df.sort_values(['rul_ga_jam', 'durasi_perbaikan_jam']).index.tolist(),
}

rows = []
for name, order in orders.items():
    result = evaluate_schedule(order, df)
    rows.append({
        'jadwal': name,
        'breakdown_count': result['breakdown_count'],
        'total_lateness_jam': round(result['total_lateness_jam'], 2),
        'total_waiting_jam': round(result['total_waiting_jam'], 2),
        'makespan_jam': round(result['makespan_jam'], 2),
        'cost': round(result['cost'], 2),
        'fitness': result['fitness'],
    })

fitness_check = pd.DataFrame(rows).sort_values('cost')
fitness_check

## 5. Operator Evolusi dan Batasan Jadwal

Batasan utama jadwal adalah setiap mesin wajib muncul tepat satu kali, tidak boleh ada mesin ganda, dan perbaikan berjalan satu per satu. Operator yang kami gunakan adalah Order Crossover (OX), swap mutation, dan inversion mutation.

In [ ]:
def is_valid_chromosome(order, n_machine):
    return len(order) == n_machine and len(set(order)) == n_machine and set(order) == set(range(n_machine))

print('Validasi kromosom awal:', is_valid_chromosome(urutan_awal, len(df)))

## Kesimpulan Tahap 2

Desain Algoritma Genetika menggunakan hasil prediksi sisa umur dari Tahap 1. Fokus penilaian jadwal adalah jumlah breakdown, sehingga jadwal yang lebih sedikit memicu breakdown mendapat nilai lebih baik. Desain ini digunakan sebagai dasar implementasi pada Tahap 3.